### Semana 5 Día 4

AutoGen Core - Distributed

Solo voy a dar un adelanto de esto!!

En parte porque no estoy seguro de cuán relevante es para ti. Si quisieras que agregue más contenido para esto, por favor házmelo saber..

In [1]:
# Importar componentes para runtime distribuido
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import Tool
from IPython.display import display, Markdown

from dotenv import load_dotenv

load_dotenv(override=True)

ALL_IN_ONE_WORKER = True

### Empezar con nuestra clase Message

In [2]:
# Definir clase Message
@dataclass
class Message:
    content: str

### Y ahora - un host para nuestro runtime distribuido

In [3]:
# Crear host para runtime distribuido
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost 

# Iniciar host para runtime distribuido
host = GrpcWorkerAgentRuntimeHost(address="localhost:50051" )
host.start()

### Vamos a reintroducir una herramienta

In [4]:
# Configurar herramienta de búsqueda con LangChain
serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="Útil para cuando necesitas buscar en Internet")
autogen_serper = LangChainToolAdapter(langchain_serper)

In [5]:
# Instrucciones para los agentes\
instruction1 = "Para ayudar a decidir si se debe utilizar AutoGen en un nuevo proyecto de agente de IA, \
investiga y responde brevemente con los motivos a favor de elegir AutoGen; las ventajas de AutoGen."

instruction2 = "Para ayudar a decidir si se debe utilizar AutoGen en un nuevo proyecto de agente de IA, \
investigue y responda brevemente con razones en contra de elegir AutoGen; las desventajas de AutoGen."

judge = "Debe tomar una decisión sobre si utilizar AutoGen para un proyecto. \
Su equipo de investigación ha presentado las siguientes razones a favor y en contra. \
Basándose únicamente en la investigación de su equipo, responda con su decisión y una breve justificación."


### Y hacer algunos Agentes

In [6]:
# Definir agentes para investigación
class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
# Definir agentes para investigación
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
# Definir agente juez   
class Judge(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client)
        
    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        message1 = Message(content=instruction1)
        message2 = Message(content=instruction2)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message1, inner_1)
        response2 = await self.send_message(message2, inner_2)
        result = f"## Pros of AutoGen:\n{response1.content}\n\n## Cons of AutoGen:\n{response2.content}\n\n"
        judgement = f"{judge}\n{result}Respond with your decision and brief explanation"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + "\n\n## Decision:\n\n" + response.chat_message.content)

In [ ]:
# Crear workers y registrar agentes
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

if ALL_IN_ONE_WORKER:

    # Crear un solo worker y registrar todos los agentes en él
    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()
    # El ID del agente juez se usará para enviarle mensajes y obtener su decisión
    await Player1Agent.register(worker, "player1", lambda: Player1Agent("player1"))
    await Player2Agent.register(worker, "player2", lambda: Player2Agent("player2"))
    await Judge.register(worker, "judge", lambda: Judge("judge"))
    # El ID del agente juez se usará para enviarle mensajes y obtener su decisión
    agent_id = AgentId("judge", "default")

else:
    # Crear múltiples workers y registrar cada agente en un worker diferente
    worker1 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker1.start()
    await Player1Agent.register(worker1, "player1", lambda: Player1Agent("player1"))

    worker2 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker2.start()
    await Player2Agent.register(worker2, "player2", lambda: Player2Agent("player2"))
    
    # El agente juez se registra en un tercer worker, pero podría estar en cualquiera de los workers anteriores
    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()
    await Judge.register(worker, "judge", lambda: Judge("judge"))
    agent_id = AgentId("judge", "default")

In [8]:
# Enviar mensaje y obtener respuesta
response = await worker.send_message(Message(content="Go!"), agent_id)

In [9]:
# Mostrar respuesta en Markdown
display(Markdown(response.content))

## Pros of AutoGen:
Las ventajas de utilizar AutoGen en un nuevo proyecto de agente de IA incluyen:

1. **Facilitación de Conversaciones**: AutoGen permite la creación de interacciones naturales entre múltiples agentes, lo que puede mejorar la experiencia del usuario y la eficiencia de las tareas.

2. **Colaboración Eficiente**: Los agentes pueden comunicarse y delegar tareas entre sí, lo que permite un enfoque más colaborativo para resolver problemas complejos.

3. **Reducción de Repeticiones**: Al tener un sistema que organiza y gestiona las interacciones, se disminuye la redundancia en las comunicaciones, optimizando así el flujo de trabajo.

4. **Flexibilidad**: AutoGen puede adaptarse a diferentes contextos y necesidades, lo que lo hace ampliamente aplicable en diversos sectores y aplicaciones.

Estas ventajas hacen de AutoGen una opción atractiva para proyectos que buscan mejorar la interacción y colaboración entre agentes de IA. 

TERMINATE

## Cons of AutoGen:
Las desventajas de utilizar AutoGen en un nuevo proyecto de agente de IA incluyen:

1. **Requiere programación avanzada**: El uso de AutoGen no es intuitivo para usuarios no técnicos, ya que implica cierto nivel de conocimiento en programación, lo que puede limitar su accesibilidad.

2. **Sin interfaz visual**: No hay una interfaz gráfica para que los no desarrolladores puedan utilizar AutoGen fácilmente, lo que puede complicar su implementación en proyectos donde se espera que diferentes usuarios interactúen con el sistema.

3. **Costo oculto en soporte y desarrollo**: Aunque es gratuito como software de código abierto, la implementación y el mantenimiento de un sistema que usa AutoGen pueden requerir un costo significativo en términos de personal técnico y soporte.

4. **Solución parcial**: AutoGen no es una solución completa y definitiva; más bien, es una base que requiere un desarrollo adicional para crear sistemas más sofisticados, lo que puede aumentar la carga de trabajo inicial.

5. **Dificultad en el manejo del estado**: No tiene un concepto nativo de "usuario a lo largo de las sesiones", lo que puede dificultar la personalización de interacciones y la construcción de experiencias más ricas para los usuarios finales.

Estas desventajas pueden hacer que la elección de AutoGen no sea ideal para todos los proyectos, especialmente aquellos que requieren una rápida implementación o que deben ser utilizados por un público más amplio y diverso. 

TERMINATE



## Decision:

Después de evaluar los pros y los contras de utilizar AutoGen para el proyecto de agente de IA, he decidido **no utilizar AutoGen** en este momento.

La razón principal es que, aunque AutoGen ofrece ventajas significativas en términos de interacción y colaboración entre agentes de IA, las desventajas relacionadas con la accesibilidad, la necesidad de conocimientos técnicos avanzados y la falta de una interfaz visual representan obstáculos importantes para una implementación efectiva. Además, el costo potencial de soporte y desarrollo podría convertir el proyecto en un esfuerzo más complejo y costoso de lo inicialmente planeado, especialmente si se busca alcanzar una variedad de usuarios que no son técnicos. 

TERMINATE

In [10]:
# Detener workers
await worker.stop()
if not ALL_IN_ONE_WORKER:
    await worker1.stop()
    await worker2.stop()

In [11]:
# Detener host
await host.stop()